In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

import mlflow
import mlflow.sklearn

import dagshub
dagshub.init(repo_owner='dkhak22', repo_name='ml-assignment-2', mlflow=True)
mlflow.set_experiment('XGBoost_Training')

RANDOM_STATE = 42


In [ ]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test  = test_transaction.merge(test_identity,  on='TransactionID', how='left')

del train_transaction, train_identity, test_transaction, test_identity

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Fraud rate: {train["isFraud"].mean():.4f}')


# Cleaning

In [ ]:
missing_pct = train.isnull().mean().sort_values(ascending=False)
print('Top 20 columns by missing %:')
print(missing_pct.head(20))


In [ ]:
MISSING_THRESHOLD = 0.5
drop_high_missing = missing_pct[missing_pct > MISSING_THRESHOLD].index.tolist()
drop_id = ['TransactionID']
DROP_COLS = list(set(drop_high_missing + drop_id))

train_clean = train.drop(columns=DROP_COLS)
test_clean  = test.drop(columns=[c for c in DROP_COLS if c in test.columns])

print(f'Columns before: {train.shape[1]}')
print(f'Dropped (>{MISSING_THRESHOLD*100:.0f}% missing): {len(drop_high_missing)}')
print(f'Columns after:  {train_clean.shape[1]}')


# Feature Engineering

In [ ]:
HIGH_CARD_COLS = [
    'card1', 'card2', 'card3', 'card5',
    'P_emaildomain', 'R_emaildomain',
    'DeviceInfo', 'id_31', 'id_33', 'id_20'
]
HIGH_CARD_COLS = [c for c in HIGH_CARD_COLS if c in train_clean.columns]


In [ ]:
class FraudFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, freq_cols=None):
        self.freq_cols = freq_cols or []

    def fit(self, X, y=None):
        X = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.freq_maps_ = {}
        for col in self.freq_cols:
            if col in X.columns:
                self.freq_maps_[col] = X[col].value_counts(normalize=True).to_dict()
        return self

    def transform(self, X):
        X = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        X = X.copy()

        if 'TransactionDT' in X.columns:
            X['tx_hour']    = (X['TransactionDT'] // 3600) % 24
            X['tx_weekday'] = (X['TransactionDT'] // (3600 * 24)) % 7
            X['tx_day']     = (X['TransactionDT'] // (3600 * 24)) % 30

        if 'TransactionAmt' in X.columns:
            X['TransactionAmt_log']   = np.log1p(X['TransactionAmt'])
            X['TransactionAmt_cents'] = X['TransactionAmt'] - np.floor(X['TransactionAmt'])

        for col in self.freq_cols:
            if col in X.columns and col in self.freq_maps_:
                X[f'{col}_freq'] = X[col].map(self.freq_maps_[col]).fillna(0)

        return X


In [ ]:
fe_inspect = FraudFeatureEngineer(freq_cols=HIGH_CARD_COLS)
fe_inspect.fit(train_clean.drop(columns=['isFraud']))
train_fe = fe_inspect.transform(train_clean.drop(columns=['isFraud']))

new_features = (
    ['tx_hour', 'tx_weekday', 'tx_day', 'TransactionAmt_log', 'TransactionAmt_cents']
    + [f'{c}_freq' for c in HIGH_CARD_COLS]
)
new_features = [c for c in new_features if c in train_fe.columns]
print(f'New features added: {new_features}')
print(f'Total features after FE: {train_fe.shape[1]}')


# Feature Selection

In [ ]:
def calculate_iv(series, target, n_bins=10):
    total_events     = target.sum()
    total_non_events = len(target) - total_events
    if total_events == 0 or total_non_events == 0:
        return 0.0
    temp = pd.DataFrame({'f': series, 't': target}).dropna()
    if temp.empty:
        return 0.0
    if temp['f'].dtype == 'object' or temp['f'].nunique() < 20:
        groups = temp.groupby('f', observed=True)['t'].agg(['sum', 'count'])
    else:
        try:
            temp['bin'] = pd.qcut(temp['f'].rank(method='first'), n_bins, duplicates='drop')
            groups = temp.groupby('bin', observed=True)['t'].agg(['sum', 'count'])
        except Exception:
            return 0.0
    groups.columns = ['events', 'total']
    groups['non_events']     = groups['total'] - groups['events']
    groups['pct_events']     = groups['events'].clip(lower=1) / total_events
    groups['pct_non_events'] = groups['non_events'].clip(lower=1) / total_non_events
    groups['woe']            = np.log(groups['pct_events'] / groups['pct_non_events'])
    groups['iv']             = (groups['pct_events'] - groups['pct_non_events']) * groups['woe']
    return groups['iv'].sum()


In [ ]:
y_train = train_clean['isFraud']

sample_idx = train_fe.sample(min(60000, len(train_fe)), random_state=RANDOM_STATE).index
X_sample = train_fe.loc[sample_idx]
y_sample = y_train.loc[sample_idx]

print('Calculating IV for all features...')
iv_scores = {}
for col in train_fe.columns:
    try:
        iv_scores[col] = calculate_iv(X_sample[col], y_sample)
    except Exception:
        iv_scores[col] = 0.0

iv_df = (pd.DataFrame.from_dict(iv_scores, orient='index', columns=['IV'])
           .sort_values('IV', ascending=False))

print('\nIV interpretation: <0.02 useless | 0.02-0.1 weak | 0.1-0.3 medium | 0.3-0.5 strong')
print(iv_df.head(30))


In [ ]:
IV_THRESHOLD = 0.02

selected_features = iv_df[iv_df['IV'] > IV_THRESHOLD].index.tolist()

NUMERIC_COLS = train_fe[selected_features].select_dtypes(include=[np.number]).columns.tolist()
CAT_COLS     = train_fe[selected_features].select_dtypes(include=['object']).columns.tolist()

print(f'Selected {len(selected_features)} features (IV > {IV_THRESHOLD})')
print(f'  Numeric:     {len(NUMERIC_COLS)}')
print(f'  Categorical: {len(CAT_COLS)}')


# Training

In [ ]:
def build_pipeline(n_estimators=100, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8):
    fe = FraudFeatureEngineer(freq_cols=HIGH_CARD_COLS)

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ])
    preprocessor = ColumnTransformer([
        ('num', num_pipe, NUMERIC_COLS),
        ('cat', cat_pipe, CAT_COLS),
    ], remainder='drop')

    # Scale positive weight for imbalanced classes
    neg = (train_clean['isFraud'] == 0).sum()
    pos = (train_clean['isFraud'] == 1).sum()
    scale_pos_weight = neg / pos

    return Pipeline([
        ('fe',    fe),
        ('prep',  preprocessor),
        ('model', XGBClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            scale_pos_weight=scale_pos_weight,
            eval_metric='auc',
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])


In [ ]:
X_train = train_clean.drop(columns=['isFraud'])
y_train = train_clean['isFraud']

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
)

param_grid = [
    {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.1,  'subsample': 0.8, 'colsample_bytree': 0.8},
    {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1,  'subsample': 0.8, 'colsample_bytree': 0.8},
    {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.7, 'colsample_bytree': 0.7},
    {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'subsample': 0.6, 'colsample_bytree': 0.6},
]

best_val_auc = -1
best_run_id  = None

for params in param_grid:
    run_name = f'XGB_n{params["n_estimators"]}_d{params["max_depth"]}_lr{params["learning_rate"]}'
    with mlflow.start_run(run_name=run_name):
        pipe = build_pipeline(**params)
        pipe.fit(X_tr, y_tr)

        train_auc   = roc_auc_score(y_tr,  pipe.predict_proba(X_tr)[:,  1])
        val_auc     = roc_auc_score(y_val, pipe.predict_proba(X_val)[:, 1])
        overfit_gap = train_auc - val_auc

        mlflow.log_params(params)
        mlflow.log_param('n_features', len(selected_features))

        mlflow.log_metric('train_auc',   train_auc)
        mlflow.log_metric('val_auc',     val_auc)
        mlflow.log_metric('overfit_gap', overfit_gap)

        if overfit_gap > 0.05:
            fit_status = 'overfit'
        elif val_auc < 0.70:
            fit_status = 'underfit'
        else:
            fit_status = 'good_fit'
        mlflow.set_tag('fit_status', fit_status)

        print(f'{run_name}  train_auc={train_auc:.4f}  val_auc={val_auc:.4f}  gap={overfit_gap:.4f}  {fit_status}')

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_params  = params

print(f'\nBest val_auc={best_val_auc:.4f}  params={best_params}')
print('Retraining best model on full data...')

with mlflow.start_run(run_name='XGB_best'):
    full_pipe = build_pipeline(**best_params)
    full_pipe.fit(X_train, y_train)
    mlflow.log_params(best_params)
    mlflow.log_metric('val_auc', best_val_auc)
    mlflow.set_tag('is_best', 'true')
    mlflow.sklearn.log_model(
        full_pipe,
        name='model',
        registered_model_name='XGBoost_FraudDetection'
    )
    best_run_id = mlflow.active_run().info.run_id

print(f'Run ID: {best_run_id}')


In [ ]:
print(f'Train AUC:    {train_auc:.4f}')
print(f'Val AUC:      {val_auc:.4f}')
print(f'Overfit gap:  {overfit_gap:.4f}')
print(f'Fit status:   {fit_status}')
